## 1. Prompt Engineering

### **n-Shot Prompting**

**0-shot:** No examples, just ask

Translate to French: "Good morning"
→ "Bonjour"

#### Code Snippet:

In [ ]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

# No examples provided - just direct instruction
prompt = "Translate to French: 'Good morning'"

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0.3
)

print(response.choices[0].message.content)
# Output: "Bonjour"


**1-shot:** One example

Translate English to French:
"Good night" → "Bonne nuit"
"Thank you" →
→ "Merci"


#### Code Snippet:

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

# One example provided to guide the model
prompt = """Translate English to French:
"Good night" → "Bonne nuit"
"Thank you" →"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0.3
)

print(response.choices[0].message.content)
# Output: "Merci"


**Few-shot:** Multiple examples

Translate English to French:
"Hello" → "Bonjour"
"Goodbye" → "Au revoir"
"Please" → "S'il vous plaît"
"See you soon" →
→ "À bientôt"


#### Code Snippet:

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

# Multiple examples to establish clear pattern
prompt = """Translate English to French:
"Hello" → "Bonjour"
"Goodbye" → "Au revoir"
"Please" → "S'il vous plaît"
"See you soon" →"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0.3
)

print(response.choices[0].message.content)
# Output: "À bientôt"


#### **Prompt Types Comparison**

| Type | Provider | Purpose | Example |
|------|----------|---------|---------|
| **System** | Developer | Sets behavior/tone | "You are a helpful assistant that replies in Markdown" |
| **User** | End User | Questions/instructions | "Summarize this paragraph" |
| **Assistant** | Model | Maintains context | Previous model responses |
| **Chain-of-Thought** | User | Step-by-step reasoning | "Let's think step by step: What is 17 + 25?" |

---

## 2. Tools: Giving LLMs Autonomy

**Layman:** Tools let the LLM DO things, not just SAY things. It's the difference between asking "What's the weather?" and actually checking weather.com to find out.

**Technical Definition:**
Functions or capabilities that an LLM can invoke to interact with external systems, perform computations, retrieve data, or take actions beyond text generation.

---

### What are Tools?

**The Core Concept:**
- LLMs generate text
- But sometimes you need to DO something
- Tools extend LLM capabilities into the real world

**Analogy:**
- **Without tools:** LLM is like a person locked in a room answering questions from memory
- **With tools:** LLM is like a person with a computer, phone, and internet - they can actually look things up and do things

---

### Common Tool Examples:

**Information Retrieval:**
- Search the web
- Query databases
- Read files
- Fetch API data

**Computation:**
- Calculate mathematical expressions
- Process data
- Run code
- Analyze spreadsheets

**Actions:**
- Send emails
- Create calendar events
- Post to social media
- Make purchases

**Communication:**
- Send messages (Slack, SMS)
- Make API calls
- Trigger webhooks
- Call other services

---

### Initial Concern vs Reality:

### The Fear:

**"OpenAI can reach into my computer?" 😱**

- Scary autonomous AI
- Direct system access
- Uncontrolled execution
- Security nightmare

---

### The Reality:

**"LLM politely suggests actions, you decide whether to execute" 😌**

**How It Actually Works:**

1. **LLM Responds with Instructions**
   - Doesn't execute anything
   - Returns structured JSON describing desired action
   - "I think we should call this function with these parameters"

2. **Your Code Decides**
   - YOU validate the request
   - YOU check permissions
   - YOU execute (or don't execute)
   - YOU control everything

3. **Complete Control**
   - Define which tools exist
   - Validate before executing
   - Log all actions
   - Implement safety checks

---

### Tool Calling: Theory vs Practice

### Theory (Sounds Scary):

**"LLM directly controls your computer"**

![theory]((https://raw.githubusercontent.com/sprashant433/GenAI/main/images/tools_theory.png)


---

### Practice (Actually Happens):

**"LLM responds with action instructions"**


![practical]((https://raw.githubusercontent.com/sprashant433/GenAI/main/images/tools_practical.png)

---

### The Tool Calling Flow:

```
1. User: "What's the weather in Boston?"
         ↓
2. LLM: "I need to call the weather tool"
         ↓
3. LLM Returns: {
    "tool": "get_weather",
    "arguments": {"city": "Boston"}
  }
         ↓
4. Your Code: Validates this request
         ↓
5. Your Code: Executes function
         ↓
6. Your Code: get_weather("Boston") → "72°F, Sunny"
         ↓
7. Your Code: Sends result back to LLM
         ↓
8. LLM: "The weather in Boston is 72°F and sunny."
         ↓
9. User sees final response
```

---

### Technical Implementation:

### Step 1: Define Tools

**Layman:** Describe to the LLM what tools are available and how to use them.

```python
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The city name"
                    },
                    "units": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Temperature units"
                    }
                },
                "required": ["city"]
            }
        }
    }
]
```

**What This JSON Says:**
- Tool name: "get_weather"
- What it does: "Get current weather for a city"
- Required parameter: "city" (a string)
- Optional parameter: "units" (celsius or fahrenheit)

---

### Step 2: LLM Decides to Use Tool

```python
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-4",
    messages=[
        {"role": "user", "content": "What's the weather in Boston?"}
    ],
    tools=tools
)

# LLM responds with tool call
tool_call = response.choices[0].message.tool_calls[0]

print(tool_call)
# Output:
# {
#   "id": "call_abc123",
#   "type": "function",
#   "function": {
#     "name": "get_weather",
#     "arguments": '{"city": "Boston", "units": "fahrenheit"}'
#   }
# }
```

**Key Point:** LLM just SUGGESTS the tool call. It doesn't execute anything.

---

### Step 3: Your Code Executes Tool

```python
import json
import requests

def get_weather(city, units="fahrenheit"):
    """Actually fetch weather data"""
    # Call real weather API
    api_key = "your_api_key"
    url = f"https://api.weather.com/v1/current?city={city}&units={units}&key={api_key}"
    response = requests.get(url)
    data = response.json()
    
    return f"{data['temperature']}°, {data['conditions']}"

# Parse tool call arguments
function_name = tool_call.function.name
function_args = json.loads(tool_call.function.arguments)

# Validate (important!)
if function_name == "get_weather":
    # Execute the actual function
    result = get_weather(**function_args)
    # Result: "72°F, Sunny"
else:
    result = "Error: Unknown function"
```

**Key Point:** YOU execute the function. YOU control what happens.

---

### Step 4: Send Result Back to LLM

```python
# Send function result back to LLM
second_response = client.chat.completions.create(
    model="gpt-4",
    messages=[
        {"role": "user", "content": "What's the weather in Boston?"},
        response.choices[0].message,  # Original response with tool call
        {
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": result  # "72°F, Sunny"
        }
    ],
    tools=tools
)

# LLM generates natural language response
final_answer = second_response.choices[0].message.content
print(final_answer)
# "The weather in Boston is currently 72°F and sunny."
```

---

### Complete Example:

```python
def chat_with_tools(user_message):
    # Initial call
    response = client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": user_message}],
        tools=tools
    )
    
    message = response.choices[0].message
    
    # Check if LLM wants to use a tool
    if message.tool_calls:
        # Execute each tool call
        tool_results = []
        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)
            
            # Execute (with validation!)
            if function_name == "get_weather":
                result = get_weather(**function_args)
            elif function_name == "search_web":
                result = search_web(**function_args)
            else:
                result = "Error: Unknown function"
            
            tool_results.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })
        
        # Send results back to LLM
        final_response = client.chat.completions.create(
            model="gpt-4",
            messages=[
                {"role": "user", "content": user_message},
                message,
                *tool_results
            ],
            tools=tools
        )
        
        return final_response.choices[0].message.content
    
    else:
        # No tool needed, return directly
        return message.content
```

---

### Security and Safety:

### 1. Validation
```python
ALLOWED_FUNCTIONS = {"get_weather", "search_web", "calculate"}

if function_name not in ALLOWED_FUNCTIONS:
    raise PermissionError(f"Function {function_name} not allowed")
```

### 2. Parameter Validation
```python
def validate_weather_params(city, units):
    if not isinstance(city, str) or len(city) > 100:
        raise ValueError("Invalid city parameter")
    
    if units not in ["celsius", "fahrenheit"]:
        raise ValueError("Invalid units parameter")
```

### 3. Rate Limiting
```python
@rate_limit(max_calls=10, period=60)
def get_weather(city, units):
    # ... implementation
```

### 4. Logging
```python
logger.info(f"Tool call: {function_name} with args {function_args}")
```

### 5. Human Approval for Critical Actions
```python
CRITICAL_ACTIONS = {"delete_data", "send_email", "make_purchase"}

if function_name in CRITICAL_ACTIONS:
    approved = await request_human_approval(function_name, function_args)
    if not approved:
        return "Action cancelled by user"
```

---

### Key Security Points:

1. **LLM Never Has Direct System Access**
   - Can only suggest actions
   - All execution controlled by you

2. **You Control Everything:**
   - Which tools exist
   - Who can use them
   - When they execute
   - What parameters are valid

3. **Tool Calling is Declarative, Not Imperative:**
   - LLM declares intent
   - Your code imperative implements

---

### Tool Call Format:

**What LLM Returns:**
```json
{
  "tool_calls": [
    {
      "id": "call_abc123",
      "type": "function",
      "function": {
        "name": "query_database",
        "arguments": "{\"query\": \"SELECT * FROM users WHERE active=true\", \"database\": \"production\"}"
      }
    }
  ]
}
```

**Not Executable Code!**
- Just JSON describing what to do
- Your code parses and decides whether to execute

---

### Remember:

**Tools ≠ Direct Computer Access**
**Tools = JSON + If Statements**

The LLM sends JSON describing what it wants to do.
Your code has if statements that decide whether to do it.

That's it. That's "tool calling."

Not scary. Very controllable. Extremely useful.


## The Simple Truth About Tools

**Layman Explanation:**
Tools sound fancy, but they're actually super simple! When you hear "AI agents with tools," don't think of complex magic. Think of this:

1. **JSON** - A text format that describes what function to call
2. **If statements** - Your code checking "should I run this function?"

That's it!

---

### Breaking It Down:

**Step 1: JSON describes the tool**
```json
{
  "name": "get_weather",
  "description": "Gets weather for a city",
  "parameters": {
    "city": "string"
  }
}
```

**What this means:** "Hey, there's a function called get_weather that needs a city name"

---

**Step 2: LLM responds with JSON**
```json
{
  "function": "get_weather",
  "arguments": {"city": "Boston"}
}
```

**What this means:** "I think you should call get_weather with Boston"

---

**Step 3: Your code uses if statements**
```python
if function_name == "get_weather":
    result = get_weather(city)
elif function_name == "search_web":
    result = search_web(query)
else:
    result = "Unknown function"
```

**What this means:** "Let me check if this is allowed and then execute it"

---

### That's All It Is!

**No magic. No AI reaching into your computer. Just:**
1. JSON describing functions
2. LLM choosing which function fits the task
3. Your if statements deciding whether to execute

**Technical Reality:**
- JSON = Data format for tool descriptions
- If statements = Your validation and execution logic
- LLM = Chooses appropriate tool based on context

**Why This Matters for Understanding:**
When you see complex agent frameworks, remember they're all doing this same basic thing under the hood. The frameworks just make it easier to:
- Define tools
- Parse LLM responses
- Execute functions safely
- Handle errors

But fundamentally? **Tools = JSON + If statements.**